# 02 — SFT warm-start (Gemma 3 4B-IT + Unsloth + TRL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneh2909/Overfitters/blob/main/notebooks/02_sft.ipynb)
[![Open Adapter](https://huggingface.co/datasets/huggingface/badges/resolve/main/open-in-hf-models-sm.svg)](https://huggingface.co/SnehShah/house-md-sft-gemma3-4b)

**Goal**: teach the base Gemma 3 4B-IT model to *output valid action JSON* before we ever hand it to GRPO. SFT is cheap, narrow, and unlocks the rest of the pipeline.

**Hardware**: tested on a Colab **T4 (16 GB)** in `~12 min` (200 samples × 1 epoch). The full production run — **2,151 oracle traces × 1 epoch on an L4** — produced the adapter at [`SnehShah/house-md-sft-gemma3-4b`](https://huggingface.co/SnehShah/house-md-sft-gemma3-4b) in ~45 min.

**What we do here**
1. Load `SnehShah/house-md-sft-data` (oracle traces in `prompt`/`completion` form).
2. 4-bit-quantise `unsloth/gemma-3-4b-it-unsloth-bnb-4bit` and attach a **LoRA-r16** adapter.
3. Train with TRL's `SFTTrainer`, masking the prompt with `train_on_responses_only` so loss flows only through the action JSON.
4. Quick format-validity smoke test against the live OpenEnv Space.
5. Push the adapter to your HF account.

> The full production training script is at [`scripts/train_sft.py`](https://github.com/sneh2909/Overfitters/blob/main/scripts/train_sft.py).

## 1. Install Unsloth + TRL

Unsloth's pre-built wheel handles the bf16 / fp16 mismatch on T4 automatically.

In [ ]:
%%capture
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets
!pip install -q huggingface_hub wandb

## 2. Authenticate with Hugging Face + W&B (optional)

Set these in **Colab Secrets** (🔑 left sidebar) so they don't end up in the notebook output:
* `HF_TOKEN` — needed to push the adapter
* `WANDB_API_KEY` — needed to log curves to W&B (optional)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY") or ""
except Exception:
    pass

HF_USERNAME   = os.environ.get("HF_USERNAME", "SnehShah")
HF_TOKEN      = os.environ.get("HF_TOKEN", "")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ logged into HF")
else:
    print("⚠  HF_TOKEN missing — push step will be skipped")

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("✓ logged into W&B")

## 3. Load the dataset

`SnehShah/house-md-sft-data` is private. If you don't have access, set `DATA_PATH = "data/sft_dataset.jsonl"` after cloning the repo — the file is shipped in the GitHub source.

We **deliberately train on a 200-sample slice** here so the loop fits comfortably in T4 VRAM and judges can re-run it without burning a Colab credit. The production checkpoint trained on all 2,151.

In [ ]:
from datasets import load_dataset, Dataset
from pathlib import Path
import json

MINI_SAMPLES = 200

if Path("data/sft_dataset.jsonl").exists():
    rows = [json.loads(l) for l in open("data/sft_dataset.jsonl")][:MINI_SAMPLES]
    ds = Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]} for r in rows])
else:
    ds = load_dataset("SnehShah/house-md-sft-data", split="train").select(range(MINI_SAMPLES))

print(f"using {len(ds)} samples")
print("\nsample completion:", ds[0]["completion"])

## 4. Load Gemma 3 4B in 4-bit + attach LoRA

Memory budget on T4 (16 GB):
* base model 4-bit: ~3.5 GB
* activations @ seq_len=2048, bsz=2: ~8 GB
* optimizer + grads (LoRA-r16, 8-bit AdamW): ~1.5 GB

If you OOM, drop `MAX_SEQ_LEN` from 2048 to 1024.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
MODEL_NAME  = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✓ LoRA-r16 attached")

## 5. Format with Gemma 3 chat template

Each `prompt` becomes a `user` turn and each `completion` becomes a `model` turn. We then tell `train_on_responses_only` to mask everything before `<start_of_turn>model\n` so loss only flows through the JSON action.

In [ ]:
def format_row(ex):
    msgs = [
        {"role": "user",      "content": ex["prompt"]},
        {"role": "assistant", "content": ex["completion"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds_fmt = ds.map(format_row, remove_columns=[c for c in ds.column_names if c not in ("prompt", "completion")])
ds_fmt = ds_fmt.remove_columns([c for c in ds_fmt.column_names if c != "text"])
print(ds_fmt[0]["text"][:400], "...")

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported, train_on_responses_only

OUTPUT_HUB_ID = f"{HF_USERNAME}/house-md-sft-gemma3-4b-mini"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_fmt,
    args=SFTConfig(
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        output_dir="outputs/sft-mini",
        save_strategy="epoch",
        logging_steps=2,
        report_to="wandb" if WANDB_API_KEY else "none",
        run_name="sft-gemma3-4b-house-md-mini",
        seed=42,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)
trainer.train()

## 7. Quick format-validity smoke check

We pull the first 5 patients from the held-out eval set and ask the freshly-tuned model for an action. We don't care about correctness here — just that the JSON parses. **Target: ≥4/5.**

In [ ]:
import json, requests, torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

ENV_URL = "https://snehshah-house-md-env.hf.space"
valid = 0
for i in range(5):
    s = requests.post(f"{ENV_URL}/api/episodes", json={}).json()
    sid = s["session_id"]
    obs = s["observation"]
    prompt = obs["prompt"]

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        j = json.loads(raw)
        if {"type", "argument"} <= set(j):
            valid += 1
            print(f"[{i+1}/5] ✓ {j['type']}({j['argument']})")
    except Exception as e:
        print(f"[{i+1}/5] ✗ parse error: {e} — raw: {raw[:120]}")
    requests.delete(f"{ENV_URL}/api/episodes/{sid}")

print(f"\nFormat validity: {valid}/5 — target ≥ 4 after a single 200-sample epoch")

## 8. Push to the Hub

Skip this if you only wanted the smoke check. With `HF_TOKEN` set, your adapter lands at `<HF_USERNAME>/house-md-sft-gemma3-4b-mini` and is ready for `03_grpo.ipynb`.

In [ ]:
if HF_TOKEN:
    model.push_to_hub(OUTPUT_HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(OUTPUT_HUB_ID, token=HF_TOKEN)
    print(f"✓ pushed to https://huggingface.co/{OUTPUT_HUB_ID}")
else:
    model.save_pretrained("outputs/sft-mini/final")
    tokenizer.save_pretrained("outputs/sft-mini/final")
    print("✓ saved locally to outputs/sft-mini/final")

## 9. Production checkpoint

If you'd rather skip training entirely and use the full production SFT for the next notebook, just point GRPO at:

```python
SFT_ADAPTER = "SnehShah/house-md-sft-gemma3-4b"
```

It was trained on all 2,151 oracle samples for 1 epoch on an L4 (~45 min). Same LoRA shape, same chat template — a drop-in replacement for the mini one above.